# 5-Fold Mean Feature Count by Dataset and Model

이 노트북은 `hyper_parameter/cv_weights`에 저장된 fold artifact를 읽어서, 각 데이터셋별로 각 모델이 실제로 사용한 feature 수를 5-fold 평균으로 집계합니다.

- 기본 집계 대상은 `mode == 'no_mi'` 입니다.
- `GH-ANFIS`는 artifact 내부 gate mask를 읽어 `base / residual / full` feature 수를 계산합니다.
- 나머지 모델은 artifact에 저장된 실제 입력 차원(`n_features`)을 사용합니다.
- `GA-ANFIS`, `PSO-ANFIS`는 설정된 `selected_features` 개수와 실제 사용된 입력 차원이 다를 수 있으므로 진단 표를 함께 출력합니다.

In [1]:
from pathlib import Path
from typing import Optional
import warnings

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.base import InconsistentVersionWarning

warnings.filterwarnings('ignore', category=InconsistentVersionWarning)

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
WEIGHT_ROOT = ROOT / 'hyper_parameter' / 'cv_weights'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODE_FILTER = 'no_mi'
DATASET_ORDER = [
    'Breast_Cancer_Wisconsin_(Original)',
    'Vowel',
    'Spambase',
    'Gisette',
]
MODEL_ORDER = [
    'GH-ANFIS(base)',
    'GH-ANFIS(residual)',
    'GH-ANFIS(full)',
    'ANFIS',
    'GA-ANFIS',
    'PSO-ANFIS',
    'PH-ANFIS(Avg)',
    'PH-ANFIS(Stacked)',
    'SVM',
]

FILE_MODEL_MAP = {
    'gh_anfis': 'GH-ANFIS',
    'anfis': 'ANFIS',
    'ga_anfis': 'GA-ANFIS',
    'pso_anfis': 'PSO-ANFIS',
    'ph_anfis_avg': 'PH-ANFIS(Avg)',
    'ph_anfis_stacked': 'PH-ANFIS(Stacked)',
    'svm': 'SVM',
}

WEIGHT_ROOT

PosixPath('/home/harp3133t/Research/03_Research/GH-ANFIS_E403/hyper_parameter/cv_weights')

In [2]:
def split_dataset_and_mode(dataset_value: str):
    dataset_value = str(dataset_value)
    if '__' not in dataset_value:
        return dataset_value, None
    dataset_name, mode = dataset_value.rsplit('__', 1)
    return dataset_name, mode


def load_artifact_payload(path: Path):
    if path.suffix == '.joblib':
        return joblib.load(path)
    return torch.load(path, map_location='cpu')


def _tensor_to_numpy(value):
    if value is None:
        return None
    if isinstance(value, np.ndarray):
        return value
    if hasattr(value, 'detach'):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def gh_feature_counts_from_payload(payload: dict):
    state_dict = payload.get('state_dict') or {}
    params = payload.get('params') or {}

    def resolve_mask(prefix: str):
        hard_key = f'{prefix}_mask_hard'
        logits_key = f'{prefix}_mask_logits'
        threshold = params.get(f'{prefix}_mask_threshold')
        threshold = 0.5 if threshold is None else float(threshold)

        if hard_key in state_dict:
            mask = _tensor_to_numpy(state_dict[hard_key]).reshape(-1)
            return mask > 0
        if logits_key in state_dict:
            logits = state_dict[logits_key].detach().cpu()
            probs = torch.sigmoid(logits).numpy().reshape(-1)
            return probs >= threshold
        return np.ones(int(payload.get('n_features', 0)), dtype=bool)

    base_mask = resolve_mask('base')
    residual_mask = resolve_mask('residual')
    residual_mode = str(params.get('residual_gate_mode', 'complement')).strip().lower()
    if residual_mode == 'complement':
        residual_effective = residual_mask & (~base_mask)
    else:
        residual_effective = residual_mask

    full_mask = base_mask | residual_effective
    return {
        'GH-ANFIS(base)': int(base_mask.sum()),
        'GH-ANFIS(residual)': int(residual_effective.sum()),
        'GH-ANFIS(full)': int(full_mask.sum()),
    }


def collect_feature_counts(weight_root: Path, mode_filter: Optional[str] = 'no_mi'):
    rows = []
    for fold_dir in sorted(weight_root.glob('*/*')):
        if not fold_dir.is_dir() or not fold_dir.name.startswith('fold_'):
            continue

        for artifact_path in sorted(fold_dir.iterdir()):
            if artifact_path.suffix not in {'.pt', '.joblib'}:
                continue

            payload = load_artifact_payload(artifact_path)
            dataset_name, mode = split_dataset_and_mode(payload.get('dataset', fold_dir.parent.name))
            if mode_filter is not None and mode != mode_filter:
                continue

            file_stem = artifact_path.stem
            base_model_name = FILE_MODEL_MAP.get(file_stem, payload.get('model_name', file_stem))
            fold_value = int(payload.get('fold', str(fold_dir.name).split('_')[-1]))
            configured_selected = payload.get('params', {}).get('selected_features')
            configured_selected_len = len(configured_selected or []) if configured_selected is not None else np.nan

            if file_stem == 'gh_anfis':
                gh_counts = gh_feature_counts_from_payload(payload)
                for gh_model_name, feature_count in gh_counts.items():
                    rows.append({
                        'dataset': dataset_name,
                        'mode': mode,
                        'fold': fold_value,
                        'model': gh_model_name,
                        'feature_count': int(feature_count),
                        'count_basis': 'gh_gate_mask',
                        'configured_selected_len': np.nan,
                        'artifact_path': str(artifact_path.relative_to(ROOT)),
                    })
                continue

            rows.append({
                'dataset': dataset_name,
                'mode': mode,
                'fold': fold_value,
                'model': base_model_name,
                'feature_count': int(payload.get('n_features', 0)),
                'count_basis': 'artifact_n_features',
                'configured_selected_len': configured_selected_len,
                'artifact_path': str(artifact_path.relative_to(ROOT)),
            })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df['dataset'] = pd.Categorical(df['dataset'], categories=DATASET_ORDER, ordered=True)
    df['model'] = pd.Categorical(df['model'], categories=MODEL_ORDER, ordered=True)
    return df.sort_values(['dataset', 'model', 'fold']).reset_index(drop=True)

In [3]:
feature_counts_df = collect_feature_counts(WEIGHT_ROOT, mode_filter=MODE_FILTER)
feature_counts_df

,dataset,mode,fold,model,feature_count,count_basis,configured_selected_len,artifact_path
0,Breast_Cancer_Wisconsin_(Original),no_mi,1,GH-ANFIS(base),26,gh_gate_mask,NaN,hyper_parameter/cv_weights/Breast_Cancer_Wisco...
1,Breast_Cancer_Wisconsin_(Original),no_mi,2,GH-ANFIS(base),35,gh_gate_mask,NaN,hyper_parameter/cv_weights/Breast_Cancer_Wisco...
2,Breast_Cancer_Wisconsin_(Original),no_mi,3,GH-ANFIS(base),33,gh_gate_mask,NaN,hyper_parameter/cv_weights/Breast_Cancer_Wisco...
3,Breast_Cancer_Wisconsin_(Original),no_mi,4,GH-ANFIS(base),42,gh_gate_mask,NaN,hyper_parameter/cv_weights/Breast_Cancer_Wisco...
4,Breast_Cancer_Wisconsin_(Original),no_mi,5,GH-ANFIS(base),29,gh_gate_mask,NaN,hyper_parameter/cv_weights/Breast_Cancer_Wisco...
...,...,...,...,...,...,...,...,...
175,Gisette,no_mi,1,SVM,5000,artifact_n_features,NaN,hyper_parameter/cv_weights/Gisette__no_mi/fold...
176,Gisette,no_mi,2,SVM,5000,artifact_n_features,NaN,hyper_parameter/cv_weights/Gisette__no_mi/fold...
177,Gisette,no_mi,3,SVM,5000,artifact_n_features,NaN,hyper_parameter/cv_weights/Gisette__no_mi/fold...
178,Gisette,no_mi,4,SVM,5000,artifact_n_features,NaN,hyper_parameter/cv_weights/Gisette__no_mi/fold...


In [4]:
feature_count_summary_df = (
    feature_counts_df
    .groupby(['dataset', 'model'], observed=True, as_index=False)
    .agg(
        n_folds=('fold', 'nunique'),
        feature_count_mean=('feature_count', 'mean'),
        feature_count_std=('feature_count', 'std'),
        feature_count_min=('feature_count', 'min'),
        feature_count_max=('feature_count', 'max'),
        count_basis=('count_basis', lambda s: ', '.join(sorted(set(map(str, s))))),
    )
    .sort_values(['dataset', 'model'])
    .reset_index(drop=True)
)
feature_count_summary_df['feature_count_std'] = feature_count_summary_df['feature_count_std'].fillna(0.0)
feature_count_summary_df

,dataset,model,n_folds,feature_count_mean,feature_count_std,feature_count_min,feature_count_max,count_basis
0,Breast_Cancer_Wisconsin_(Original),GH-ANFIS(base),5,33.0,6.123724,26,42,gh_gate_mask
1,Breast_Cancer_Wisconsin_(Original),GH-ANFIS(residual),5,42.8,7.293833,32,49,gh_gate_mask
2,Breast_Cancer_Wisconsin_(Original),GH-ANFIS(full),5,75.8,2.049390,74,78,gh_gate_mask
3,Breast_Cancer_Wisconsin_(Original),ANFIS,5,80.0,0.000000,80,80,artifact_n_features
4,Breast_Cancer_Wisconsin_(Original),GA-ANFIS,5,80.0,0.000000,80,80,artifact_n_features
5,Breast_Cancer_Wisconsin_(Original),PSO-ANFIS,5,80.0,0.000000,80,80,artifact_n_features
6,Breast_Cancer_Wisconsin_(Original),PH-ANFIS(Avg),5,80.0,0.000000,80,80,artifact_n_features
7,Breast_Cancer_Wisconsin_(Original),PH-ANFIS(Stacked),5,80.0,0.000000,80,80,artifact_n_features
8,Breast_Cancer_Wisconsin_(Original),SVM,5,80.0,0.000000,80,80,artifact_n_features
9,Vowel,GH-ANFIS(base),5,27.0,0.000000,27,27,gh_gate_mask


In [5]:
selection_diagnostics_df = (
    feature_counts_df
    .loc[feature_counts_df['model'].isin(['ANFIS', 'GA-ANFIS', 'PSO-ANFIS'])]
    .assign(configured_selected_len=lambda df: df['configured_selected_len'].fillna(0).astype(int))
    .assign(selection_gap=lambda df: df['feature_count'] - df['configured_selected_len'])
    .sort_values(['dataset', 'model', 'fold'])
    .reset_index(drop=True)
)
selection_diagnostics_df

,dataset,mode,fold,model,feature_count,count_basis,configured_selected_len,artifact_path,selection_gap
0,Breast_Cancer_Wisconsin_(Original),no_mi,1,ANFIS,80,artifact_n_features,0,hyper_parameter/cv_weights/Breast_Cancer_Wisco...,80
1,Breast_Cancer_Wisconsin_(Original),no_mi,2,ANFIS,80,artifact_n_features,0,hyper_parameter/cv_weights/Breast_Cancer_Wisco...,80
2,Breast_Cancer_Wisconsin_(Original),no_mi,3,ANFIS,80,artifact_n_features,0,hyper_parameter/cv_weights/Breast_Cancer_Wisco...,80
3,Breast_Cancer_Wisconsin_(Original),no_mi,4,ANFIS,80,artifact_n_features,0,hyper_parameter/cv_weights/Breast_Cancer_Wisco...,80
4,Breast_Cancer_Wisconsin_(Original),no_mi,5,ANFIS,80,artifact_n_features,0,hyper_parameter/cv_weights/Breast_Cancer_Wisco...,80
5,Breast_Cancer_Wisconsin_(Original),no_mi,1,GA-ANFIS,80,artifact_n_features,5,hyper_parameter/cv_weights/Breast_Cancer_Wisco...,75
6,Breast_Cancer_Wisconsin_(Original),no_mi,2,GA-ANFIS,80,artifact_n_features,5,hyper_parameter/cv_weights/Breast_Cancer_Wisco...,75
7,Breast_Cancer_Wisconsin_(Original),no_mi,3,GA-ANFIS,80,artifact_n_features,5,hyper_parameter/cv_weights/Breast_Cancer_Wisco...,75
8,Breast_Cancer_Wisconsin_(Original),no_mi,4,GA-ANFIS,80,artifact_n_features,5,hyper_parameter/cv_weights/Breast_Cancer_Wisco...,75
9,Breast_Cancer_Wisconsin_(Original),no_mi,5,GA-ANFIS,80,artifact_n_features,5,hyper_parameter/cv_weights/Breast_Cancer_Wisco...,75


In [6]:
selection_gap_summary_df = (
    selection_diagnostics_df
    .groupby(['dataset', 'model'], observed=True, as_index=False)
    .agg(
        actual_feature_mean=('feature_count', 'mean'),
        configured_selected_mean=('configured_selected_len', 'mean'),
        selection_gap_mean=('selection_gap', 'mean'),
    )
    .sort_values(['dataset', 'model'])
    .reset_index(drop=True)
)
selection_gap_summary_df

,dataset,model,actual_feature_mean,configured_selected_mean,selection_gap_mean
0,Breast_Cancer_Wisconsin_(Original),ANFIS,80.0,0.0,80.0
1,Breast_Cancer_Wisconsin_(Original),GA-ANFIS,80.0,5.0,75.0
2,Breast_Cancer_Wisconsin_(Original),PSO-ANFIS,80.0,7.0,73.0
3,Vowel,ANFIS,29.0,0.0,29.0
4,Vowel,GA-ANFIS,29.0,23.0,6.0
5,Vowel,PSO-ANFIS,29.0,19.0,10.0
6,Spambase,ANFIS,57.0,0.0,57.0
7,Spambase,GA-ANFIS,39.0,39.0,0.0
8,Spambase,PSO-ANFIS,31.0,31.0,0.0
9,Gisette,ANFIS,5000.0,0.0,5000.0


In [7]:
available_models = feature_count_summary_df['model'].dropna().astype(str).unique().tolist()
feature_count_pivot_mean = (
    feature_count_summary_df
    .pivot(index='dataset', columns='model', values='feature_count_mean')
    .reindex(index=DATASET_ORDER)
    .reindex(columns=[model for model in MODEL_ORDER if model in available_models])
)
feature_count_pivot_mean.round(1)

model,GH-ANFIS(base),GH-ANFIS(residual),GH-ANFIS(full),ANFIS,GA-ANFIS,PSO-ANFIS,PH-ANFIS(Avg),PH-ANFIS(Stacked),SVM
dataset,,,,,,,,,
Breast_Cancer_Wisconsin_(Original),33.0,42.8,75.8,80.0,80.0,80.0,80.0,80.0,80.0
Vowel,27.0,0.0,27.0,29.0,29.0,29.0,29.0,29.0,29.0
Spambase,25.2,25.0,50.2,57.0,39.0,31.0,57.0,57.0,57.0
Gisette,2408.8,272.0,2680.8,5000.0,91.0,87.0,5000.0,5000.0,5000.0


In [8]:
feature_count_pivot_mean_std = (
    feature_count_summary_df
    .assign(mean_std=lambda df: df.apply(lambda row: f"{row['feature_count_mean']:.1f} ± {row['feature_count_std']:.1f}", axis=1))
    .pivot(index='dataset', columns='model', values='mean_std')
    .reindex(index=DATASET_ORDER)
    .reindex(columns=[model for model in MODEL_ORDER if model in available_models])
)
feature_count_pivot_mean_std

model,GH-ANFIS(base),GH-ANFIS(residual),GH-ANFIS(full),ANFIS,GA-ANFIS,PSO-ANFIS,PH-ANFIS(Avg),PH-ANFIS(Stacked),SVM
dataset,,,,,,,,,
Breast_Cancer_Wisconsin_(Original),33.0 ± 6.1,42.8 ± 7.3,75.8 ± 2.0,80.0 ± 0.0,80.0 ± 0.0,80.0 ± 0.0,80.0 ± 0.0,80.0 ± 0.0,80.0 ± 0.0
Vowel,27.0 ± 0.0,0.0 ± 0.0,27.0 ± 0.0,29.0 ± 0.0,29.0 ± 0.0,29.0 ± 0.0,29.0 ± 0.0,29.0 ± 0.0,29.0 ± 0.0
Spambase,25.2 ± 2.6,25.0 ± 3.4,50.2 ± 2.2,57.0 ± 0.0,39.0 ± 0.0,31.0 ± 0.0,57.0 ± 0.0,57.0 ± 0.0,57.0 ± 0.0
Gisette,2408.8 ± 23.5,272.0 ± 17.4,2680.8 ± 33.9,5000.0 ± 0.0,91.0 ± 0.0,87.0 ± 0.0,5000.0 ± 0.0,5000.0 ± 0.0,5000.0 ± 0.0


In [9]:
summary_csv_path = OUTPUT_DIR / f'feature_count_summary_{MODE_FILTER or "all"}.csv'
pivot_csv_path = OUTPUT_DIR / f'feature_count_pivot_mean_{MODE_FILTER or "all"}.csv'
diagnostics_csv_path = OUTPUT_DIR / f'feature_count_diagnostics_{MODE_FILTER or "all"}.csv'

feature_count_summary_df.to_csv(summary_csv_path, index=False)
feature_count_pivot_mean.to_csv(pivot_csv_path)
selection_diagnostics_df.to_csv(diagnostics_csv_path, index=False)

print('saved:', summary_csv_path)
print('saved:', pivot_csv_path)
print('saved:', diagnostics_csv_path)

saved: /home/harp3133t/Research/03_Research/GH-ANFIS_E403/output/feature_count_summary_no_mi.csv
saved: /home/harp3133t/Research/03_Research/GH-ANFIS_E403/output/feature_count_pivot_mean_no_mi.csv
saved: /home/harp3133t/Research/03_Research/GH-ANFIS_E403/output/feature_count_diagnostics_no_mi.csv
